In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- Dataset and Utility Functions ---
class BrainTumorDataset(Dataset):
    """
    Expects structure:
      data_root/
         Training/
             ClassA/
             ClassB/
         Testing/
             ClassA/
             ClassB/
    Each class folder has .png/.jpg images.
    Loads grayscale images, converted to 3-channel for LpCNN.
    """
    def __init__(self, root_dir, subset="Training", transform=None):
        super().__init__()
        self.root_dir = root_dir
        self.subset = subset
        self.transform = transform

        self.data_path = os.path.join(self.root_dir, self.subset)
        if not os.path.exists(self.data_path):
            raise FileNotFoundError(f"{self.data_path} does not exist.")

        self.classes = sorted([
            d for d in os.listdir(self.data_path)
            if os.path.isdir(os.path.join(self.data_path, d))
        ])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(self.data_path, class_name)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    full_path = os.path.join(class_dir, fname)
                    label_idx = self.class_to_idx[class_name]
                    self.samples.append((full_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('L')  # Load as grayscale
        if self.transform:
            image = self.transform(image)
        return image, label

def count_images_in_classes(folder: str) -> dict:
    """
    Return a dict {class_name: number_of_images} for each subfolder in 'folder'.
    Used to check class distribution and balance.
    """
    if not os.path.isdir(folder):
        return {}
    result = {}
    for cls in os.listdir(folder):
        cpath = os.path.join(folder, cls)
        if os.path.isdir(cpath):
            n = sum(
                f.lower().endswith(('.png', '.jpg', '.jpeg'))
                for f in os.listdir(cpath)
            )
            result[cls] = n
    return result

def find_image_in_subfolders(root_dir: str, image_name: str) -> str:
    """
    Search each subfolder of root_dir for image_name.
    Return the full path if found, else None.
    """
    if not os.path.isdir(root_dir):
        return None
    for subdir in os.listdir(root_dir):
        cpath = os.path.join(root_dir, subdir)
        if os.path.isdir(cpath):
            candidate = os.path.join(cpath, image_name)
            if os.path.isfile(candidate):
                return candidate
    return None

def load_image_as_grayscale(path: str) -> Image.Image:
    """
    Open an image at path as grayscale PIL image.
    Returns None if fails.
    """
    try:
        return Image.open(path).convert('L')
    except:
        return None

# --- LpCNN Model ---
class LpConv2d(nn.Module):
    """
    Implements the Lp-convolution layer as described in the paper. Applies a channel-wise
    L_p-mask derived from the multivariate p-generalized normal distribution (MPND).
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, 
                 p_init=2.0, sigma_init=None):
        super(LpConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride
        self.padding = padding
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, *self.kernel_size))
        self.p = nn.Parameter(torch.full((out_channels,), float(p_init)))
        if sigma_init is None:
            sigma_init = self.kernel_size[0] / 2
        L_diag = torch.full((out_channels, 2), 1/sigma_init)
        L_offdiag = torch.zeros(out_channels, 1)
        self.L_diag = nn.Parameter(L_diag)
        self.L_offdiag = nn.Parameter(L_offdiag)
        self.register_buffer('offsets', self._create_offset_grid())
    
    def _create_offset_grid(self):
        kh, kw = self.kernel_size
        h = torch.linspace(0, kh-1, kh) - (kh-1)/2
        w = torch.linspace(0, kw-1, kw) - (kw-1)/2
        h, w = torch.meshgrid(h, w, indexing='ij')
        offsets = torch.stack([h, w], dim=0)
        offsets = (offsets - offsets.min()) / (offsets.max() - offsets.min())
        return offsets.unsqueeze(0)
    
    def _build_covariance_matrix(self):
        L = torch.zeros(self.out_channels, 2, 2, device=self.L_diag.device)
        L[:, 0, 0] = self.L_diag[:, 0]
        L[:, 1, 1] = self.L_diag[:, 1]
        L[:, 1, 0] = self.L_offdiag[:, 0]
        C = torch.bmm(L, L.transpose(1, 2))
        return C
    
    def _compute_lp_mask(self):
        p = torch.clamp(self.p, min=1.0)
        C = self._build_covariance_matrix()
        offsets = self.offsets.expand(self.out_channels, -1, -1, -1)
        C_delta_s = torch.einsum('oij,ojhw->oihw', C, offsets)
        lp_norm = torch.zeros(self.out_channels, self.kernel_size[0], self.kernel_size[1], 
                            device=C_delta_s.device)
        for o in range(self.out_channels):
            abs_val = torch.abs(C_delta_s[o])
            powered = torch.pow(abs_val, p[o])
            summed = torch.sum(powered, dim=0)
            lp_norm[o] = torch.pow(summed, p[o])
        mask = torch.exp(-lp_norm)
        return mask.unsqueeze(1)
    
    def forward(self, x):
        lp_mask = self._compute_lp_mask()
        masked_weight = self.weight * lp_mask
        return F.conv2d(x, masked_weight, stride=self.stride, padding=self.padding)

class LpCNN(nn.Module):
    """
    A simple CNN with two LpConv2d layers for brain tumor classification.
    Expects 3-channel inputs (converted from grayscale MRI images).
    """
    def __init__(self, num_classes=4):
        super(LpCNN, self).__init__()
        self.conv1 = LpConv2d(3, 64, kernel_size=7, padding=3, p_init=2.0)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = LpConv2d(64, 128, kernel_size=7, padding=3, p_init=2.0)
        self.bn2 = nn.BatchNorm2d(128)
        self.fc = nn.Linear(128 * 8 * 8, num_classes)  # For 32x32 input

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# --- Training Pipeline ---
def init_weights(m):
    """
    Initialize model weights using Kaiming for convolutions and Xavier for linear layers.
    """
    if isinstance(m, (nn.Conv2d, LpConv2d)):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    elif isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0)

def main():
    # Dataset paths (update to your dataset location)
    data_root = 'C:\\Users\\jxmra\\Downloads\\team09-Rafals\\team09-Rafals\\data\\archive'  # Contains Training/ and Testing/ folders
    train_path = os.path.join(data_root, 'Training')
    test_path = os.path.join(data_root, 'Testing')

    # Check class distribution
    try:
        train_counts = count_images_in_classes(train_path)
        test_counts = count_images_in_classes(test_path)
        print("Training class distribution:", train_counts)
        print("Testing class distribution:", test_counts)
    except Exception as e:
        print(f"Error checking class distribution: {e}")
        return

    # Define transforms
    try:
        train_transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),  # Grayscale to 3-channel
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Explicit mean and std
        ])
        test_transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    except Exception as e:
        print(f"Error defining transforms: {e}")
        return

    # Create datasets
    try:
        train_dataset = BrainTumorDataset(data_root, subset="Training", transform=train_transform)
        test_dataset = BrainTumorDataset(data_root, subset="Testing", transform=test_transform)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32)
    except Exception as e:
        print(f"Error loading datasets: {e}")
        return

    # Model setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    try:
        model = LpCNN(num_classes=len(train_dataset.classes)).to(device)
        model.apply(init_weights)
    except Exception as e:
        print(f"Error initializing model: {e}")
        return

    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10)
    num_epochs = 50
    best_val_acc = 0

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        try:
            for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)
        except Exception as e:
            print(f"Error during training: {e}")
            continue

        # Validation (using test set as proxy; ideally use a separate validation set)
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        try:
            with torch.no_grad():
                for images, labels in test_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            val_loss /= len(test_loader)
            val_acc = 100 * correct / total
            scheduler.step(val_loss)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), 'best_lp_cnn_tumor.pth')

            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, '
                  f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        except Exception as e:
            print(f"Error during validation: {e}")
            continue

    # Final evaluation
    try:
        model.load_state_dict(torch.load('best_lp_cnn_tumor.pth'))
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())
        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        print(f'\nFinal Test Results:')
        print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}')
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return

    # # Example inference on a single image
    # image_name = 'example_image.jpg'  # Replace with actual image name
    # image_path = find_image_in_subfolders(os.path.join(data_root, 'Testing'), image_name)
    # if image_path:
    #     image = load_image_as_grayscale(image_path)
    #     if image:
    #         try:
    #             image = test_transform(image).unsqueeze(0).to(device)
    #             model.eval()
    #             with torch.no_grad():
    #                 output = model(image)
    #                 _, predicted = torch.max(output, 1)
    #                 predicted_class = train_dataset.classes[predicted.item()]
    #                 print(f'Inference on {image_name}: Predicted class: {predicted_class}')
    #         except Exception as e:
    #             print(f'Error during inference: {e}')
    #     else:
    #         print(f'Failed to load {image_name}')
    # else:
    #     print(f'Image {image_name} not found')

if __name__ == "__main__":
    main()



Training class distribution: {'glioma': 1321, 'meningioma': 1339, 'notumor': 1595, 'pituitary': 1457}
Testing class distribution: {'glioma': 299, 'meningioma': 304, 'notumor': 381, 'pituitary': 300}


Epoch 1/50: 100%|██████████| 179/179 [01:54<00:00,  1.56it/s]


Epoch 1/50, Train Loss: 1.2417, Val Loss: 0.7778, Val Acc: 70.40%


Epoch 2/50: 100%|██████████| 179/179 [01:12<00:00,  2.45it/s]


Epoch 2/50, Train Loss: 0.6512, Val Loss: 1.2223, Val Acc: 60.67%


Epoch 3/50: 100%|██████████| 179/179 [01:15<00:00,  2.36it/s]


... (91 outputs left out)

Epoch 49/50: 100%|██████████| 179/179 [01:07<00:00,  2.64it/s]


Epoch 49/50, Train Loss: 0.0204, Val Loss: 0.0508, Val Acc: 98.68%


Epoch 50/50: 100%|██████████| 179/179 [01:08<00:00,  2.60it/s]


Epoch 50/50, Train Loss: 0.0189, Val Loss: 0.0511, Val Acc: 98.13%

Final Test Results:
Accuracy: 0.9868, Precision: 0.9870, Recall: 0.9868, F1: 0.9868


## GRAD-CAM SETUP

In [2]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
import numpy as np
import os

# --- Reuse Dataset and Transform ---
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load test dataset
data_root = 'C:\\Users\\jxmra\\Downloads\\team09-Rafals\\team09-Rafals\\data\\archive'
try:
    test_dataset = BrainTumorDataset(data_root, subset="Testing", transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
except Exception as e:
    print(f"Error loading test dataset: {e}")
    raise

# --- Load Trained Model ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LpCNN(num_classes=len(test_dataset.classes)).to(device)
try:
    model.load_state_dict(torch.load('best_lp_cnn_tumor.pth'))
    model.eval()
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# --- Grad-CAM Setup ---
target_layer = model.conv2
cam = GradCAM(model=model, target_layers=[target_layer])

# --- Visualization Function ---
def save_gradcam_visualization(input_image, grayscale_cam, predicted_class, true_class, output_path):
    """
    Save Grad-CAM heatmap overlaid on the original image.
    """
    # Remove batch dimension and convert to NumPy
    input_image = input_image.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)  # [1, C, H, W] -> [H, W, C]
    input_image = (input_image - input_image.min()) / (input_image.max() - input_image.min())  # Normalize to [0, 1]
    visualization = show_cam_on_image(input_image, grayscale_cam, use_rgb=True)
    plt.figure(figsize=(8, 8))
    plt.imshow(visualization)
    plt.title(f'Pred: {predicted_class}, True: {true_class}')
    plt.axis('off')
    plt.savefig(output_path, bbox_inches='tight')
    plt.close()

# --- Generate Grad-CAM for Test Images ---
try:
    os.makedirs('gradcam_outputs', exist_ok=True)
    num_images_to_visualize = 5
    visualized = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        # Inference without gradients
        with torch.no_grad():
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
        
        # Grad-CAM with gradients
        for i in range(images.size(0)):
            if visualized >= num_images_to_visualize:
                break
            input_image = images[i:i+1].clone()  # [1, C, H, W]
            input_image.requires_grad_(True)  # Enable gradient tracking
            target_class = predicted[i].item()
            predicted_class_name = test_dataset.classes[target_class]
            true_class_name = test_dataset.classes[labels[i].item()]
            img_path = test_dataset.samples[visualized][0]

            # Generate Grad-CAM heatmap
            grayscale_cam = cam(input_tensor=input_image, targets=None)[0, :]
            output_path = os.path.join('gradcam_outputs', 
                                     f'gradcam_{os.path.basename(img_path)}_{predicted_class_name}.png')
            save_gradcam_visualization(input_image, grayscale_cam, 
                                      predicted_class_name, true_class_name, output_path)
            print(f'Saved Grad-CAM visualization: {output_path}')
            visualized += 1

        if visualized >= num_images_to_visualize:
            break
except Exception as e:
    print(f"Error during Grad-CAM visualization: {e}")

# --- Grad-CAM for a Single Image ---
image_name = 'Te-no_0010.jpg'
image_path = find_image_in_subfolders(os.path.join(data_root, 'Testing'), image_name)
if image_path:
    image = load_image_as_grayscale(image_path)
    if image:
        try:
            input_image = test_transform(image).unsqueeze(0).to(device)
            # Inference without gradients
            with torch.no_grad():
                output = model(input_image)
                _, predicted = torch.max(output, 1)
                predicted_class = test_dataset.classes[predicted.item()]
                print(f'Inference on {image_name}: Predicted class: {predicted_class}')
            
            # Grad-CAM with gradients
            input_image = input_image.clone()
            input_image.requires_grad_(True)
            grayscale_cam = cam(input_tensor=input_image, targets=None)[0, :]
            output_path = os.path.join('gradcam_outputs', f'gradcam_{image_name}.png')
            save_gradcam_visualization(input_image, grayscale_cam, 
                                      predicted_class, 'Unknown', output_path)
            print(f'Saved Grad-CAM visualization: {output_path}')
        except Exception as e:
            print(f'Error during single image inference: {e}')
    else:
        print(f'Failed to load {image_name}')
else:
    print(f'Image {image_name} not found')

Saved Grad-CAM visualization: gradcam_outputs\gradcam_Te-glTr_0000.jpg_glioma.png
Saved Grad-CAM visualization: gradcam_outputs\gradcam_Te-glTr_0001.jpg_glioma.png
Saved Grad-CAM visualization: gradcam_outputs\gradcam_Te-glTr_0002.jpg_glioma.png
Saved Grad-CAM visualization: gradcam_outputs\gradcam_Te-glTr_0003.jpg_glioma.png
Saved Grad-CAM visualization: gradcam_outputs\gradcam_Te-glTr_0004.jpg_glioma.png
Inference on Te-no_0010.jpg: Predicted class: notumor
Saved Grad-CAM visualization: gradcam_outputs\gradcam_Te-no_0010.jpg.png
